In [30]:
import os
import json
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ====== PATHS: ADJUST IF NEEDED ======
FT_MODEL_DIR      = "qwen_spider_full_sft_v3"

SPIDER_DEV_JSON   = "spider_data/spider_data/dev.json"
SPIDER_TABLES_JSON= "spider_data/spider_data/tables.json"
SPIDER_DB_DIR     = "spider_data/spider_data/database"

EVAL_DIR          = "/teamspace/studios/this_studio/spider_official_eval"
NLTK_DATA_DIR     = "nltk_punkt_data"

# Gold & pred files (will be created in EVAL_DIR)
GOLD_TXT_PATH     = os.path.join(EVAL_DIR, "gold_dev.txt")
PRED_FT_PATH      = os.path.join(EVAL_DIR, "pred_ft_v3_dev.txt")

MAX_NEW_TOKENS    = 256
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

# Use your downloaded NLTK data; do NOT import nltk here
os.environ["NLTK_DATA"] = NLTK_DATA_DIR
os.environ["TRANSFORMERS_VERBOSITY"] = "info"

print("DEVICE:", DEVICE)
print("EVAL_DIR:", EVAL_DIR)
print("NLTK_DATA:", os.environ["NLTK_DATA"])


DEVICE: cuda
EVAL_DIR: /teamspace/studios/this_studio/spider_official_eval
NLTK_DATA: nltk_punkt_data


In [21]:
# Load Spider dev
with open(SPIDER_DEV_JSON, "r", encoding="utf-8") as f:
    spider_dev = json.load(f)

print("Dev examples:", len(spider_dev))
print("Example 0:", spider_dev[0]["question"], "| DB:", spider_dev[0]["db_id"])

# Write gold_dev.txt : "gold_sql<TAB>db_id"
os.makedirs(EVAL_DIR, exist_ok=True)
with open(GOLD_TXT_PATH, "w", encoding="utf-8") as out_f:
    for ex in spider_dev:
        gold_sql = ex["query"].strip()
        db_id = ex["db_id"]
        out_f.write(f"{gold_sql}\t{db_id}\n")

print("Wrote gold file:", GOLD_TXT_PATH)

# Load tables.json and build db_id -> schema string for prompts
with open(SPIDER_TABLES_JSON, "r", encoding="utf-8") as f:
    spider_tables = json.load(f)

db_schemas = {}
for db in spider_tables:
    db_id = db["db_id"]
    tables = db["table_names_original"]
    columns = db["column_names_original"]
    types = db["column_types"]

    table_cols = {i: [] for i in range(len(tables))}
    for (tbl_idx, col_name), col_type in zip(columns, types):
        if tbl_idx == -1:
            continue
        table_cols[tbl_idx].append((col_name, col_type))

    parts = []
    for i, tname in enumerate(tables):
        cols = table_cols[i]
        col_str = ", ".join(f"{c} ({ctype})" for c, ctype in cols)
        parts.append(f"{tname}({col_str})")

    db_schemas[db_id] = " | ".join(parts)

list(db_schemas.items())[:2]


Dev examples: 1034
Example 0: How many singers do we have? | DB: concert_singer
Wrote gold file: spider_official_eval/gold_dev.txt


[('perpetrator',
  'perpetrator(Perpetrator_ID (number), People_ID (number), Date (text), Year (number), Location (text), Country (text), Killed (number), Injured (number)) | people(People_ID (number), Name (text), Height (number), Weight (number), Home Town (text))'),
 ('college_2',
  'classroom(building (text), room_number (text), capacity (number)) | department(dept_name (text), building (text), budget (number)) | course(course_id (text), title (text), dept_name (text), credits (number)) | instructor(ID (text), name (text), dept_name (text), salary (number)) | section(course_id (text), sec_id (text), semester (text), year (number), building (text), room_number (text), time_slot_id (text)) | teaches(ID (text), course_id (text), sec_id (text), semester (text), year (number)) | student(ID (text), name (text), dept_name (text), tot_cred (number)) | takes(ID (text), course_id (text), sec_id (text), semester (text), year (number), grade (text)) | advisor(s_ID (text), i_ID (text)) | time_s

In [22]:
def load_tok(model_path: str):
    # try fix_mistral_regex to silence that warning; fallback if not supported
    try:
        tok = AutoTokenizer.from_pretrained(
            model_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        tok = AutoTokenizer.from_pretrained(
            model_path,
            trust_remote_code=True,
        )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

print("Loading FT v3 model...")
ft_tokenizer = load_tok(FT_MODEL_DIR)
ft_model = AutoModelForCausalLM.from_pretrained(
    FT_MODEL_DIR,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
ft_model.eval()

print("FT model loaded on", DEVICE)


Loading FT v3 model...
FT model loaded on cuda


In [23]:
SYSTEM_PROMPT = (
    "You are a text-to-SQL model. "
    "Given a natural language question and the database schema, "
    "output ONLY the SQL query that answers the question. No explanation."
)

def build_chat_messages(question: str, db_id: str):
    schema = db_schemas.get(db_id, "")
    user_content = (
        f"Database schema:\n{schema}\n\n"
        f"Question: {question}\n\n"
        f"Write the SQL query."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

def clean_for_eval(text: str) -> str:
    text = text.strip()
    # strip ``` fences if present
    if text.startswith("```"):
        lines = text.splitlines()
        if len(lines) >= 2:
            if lines[0].strip().startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            text = "\n".join(lines).strip()
    # keep from first SELECT onward
    lower = text.lower()
    sel_idx = lower.find("select")
    if sel_idx != -1:
        text = text[sel_idx:]
    # strip stray backticks/spaces
    text = text.strip("`").strip()
    return text

@torch.no_grad()
def generate_sql_for_example(ex, model, tokenizer):
    messages = build_chat_messages(ex["question"], ex["db_id"])
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(DEVICE)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,                 # greedy
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    gen_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    sql = clean_for_eval(raw_text)
    return sql


In [24]:
print("Generating FT v3 predictions for dev set...")

with open(PRED_FT_PATH, "w", encoding="utf-8") as f_ft:
    for ex in tqdm(spider_dev, dynamic_ncols=True):
        ft_sql = generate_sql_for_example(ex, ft_model, ft_tokenizer)
        f_ft.write(ft_sql.strip() + "\n")

print("Wrote FT v3 preds to:", PRED_FT_PATH)


Generating FT v3 predictions for dev set...


100%|██████████| 1034/1034 [18:04<00:00,  1.05s/it]

Wrote FT v3 preds to: spider_official_eval/pred_ft_v3_dev.txt


In [35]:
import subprocess
import os
import nltk
nltk.download('punkt_tab')


# we’re already in: /teamspace/studios/this_studio/spider_official_eval
print("Current dir:", os.getcwd())
print("Files here:", os.listdir("."))

eval_cmd_ft = [
    "python3", "evaluation.py",
    "--gold", "gold_dev.txt",
    "--pred", "pred_ft_v3_dev.txt",
    "--db",   "../spider_data/spider_data/database",
    "--table","../spider_data/spider_data/tables.json",
    "--etype","all",
]

print("=== FT v3 model evaluation ===")
print("Running:", " ".join(eval_cmd_ft))

subprocess.run(eval_cmd_ft, env=os.environ)



[nltk_data] Downloading package punkt_tab to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Current dir: /teamspace/studios/this_studio/spider_official_eval
Files here: ['pred_ft_v3_dev.txt', 'process_sql.py', 'gold_dev.txt', '__pycache__', 'evaluation.py', 'pred_base_dev.txt']
=== FT v3 model evaluation ===
Running: python3 evaluation.py --gold gold_dev.txt --pred pred_ft_v3_dev.txt --db ../spider_data/spider_data/database --table ../spider_data/spider_data/tables.json --etype all
eval_err_num:1
medium pred: SELECT s.Name, sc.Song_Release_year FROM singer s JOIN singer_in_concert sc ON s.Singer_ID = sc.Singer_ID ORDER BY s.Age ASC LIMIT 1;
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:2
medium pred: SELECT s.Name, sc.Song_Release_year FROM singer s JOIN singer_in_concert sc ON s.Singer_ID = sc.Singer_ID ORDER BY s.Age ASC LIMIT 1;
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:3
easy pred: SELECT DISTINCT c.Country FROM singer s JOIN country c ON s.Country = c.Country WHERE

CompletedProcess(args=['python3', 'evaluation.py', '--gold', 'gold_dev.txt', '--pred', 'pred_ft_v3_dev.txt', '--db', '../spider_data/spider_data/database', '--table', '../spider_data/spider_data/tables.json', '--etype', 'all'], returncode=0)